In [1]:
import pandas as pd
import sqlite3
import datetime as dt

# Load the raw data file
file_path = "online_retail_II.csv"
raw_df = pd.read_csv(file_path, encoding='ISO-8859-1')

# Set up a local SQLite in-memory database
conn = sqlite3.connect(':memory:')
raw_df.to_sql('retail_transactions', conn, index=False)

# Clean data and calculate financial baseline using SQL
query = """
    SELECT 
        Invoice, 
        StockCode, 
        Description, 
        Quantity, 
        InvoiceDate, 
        Price, 
        "Customer ID", 
        Country,
        (Quantity * Price) AS Revenue
    FROM retail_transactions
    WHERE "Customer ID" IS NOT NULL 
      AND Quantity > 0
"""

# Pull the cleaned data back into a Pandas DataFrame
df = pd.read_sql_query(query, conn)

# Fix dates for subsequent RFM and Cohort analysis
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print("--- Data Info ---")
print(df.info())

--- Data Info ---
<class 'pandas.DataFrame'>
RangeIndex: 186406 entries, 0 to 186405
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      186406 non-null  str           
 1   StockCode    186406 non-null  str           
 2   Description  186406 non-null  str           
 3   Quantity     186406 non-null  int64         
 4   InvoiceDate  186406 non-null  datetime64[us]
 5   Price        186406 non-null  float64       
 6   Customer ID  186406 non-null  float64       
 7   Country      186405 non-null  str           
 8   Revenue      186406 non-null  float64       
dtypes: datetime64[us](1), float64(3), int64(1), str(4)
memory usage: 12.8 MB
None


In [2]:
# Establish a snapshot date (one day after the latest transaction)
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

# Aggregate metrics per customer
rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency: days since last purchase
    'Invoice': 'nunique',                                     # Frequency: total distinct orders
    'Revenue': 'sum'                                          # Monetary: total money spent
}).rename(columns={
    'InvoiceDate': 'Recency', 
    'Invoice': 'Frequency', 
    'Revenue': 'Monetary'
})

# Assign quartile scores from 1 (worst) to 4 (best)
# Recency is scored inversely: fewer days since last purchase equals a higher score (4)
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=4, labels=[4, 3, 2, 1])

# Frequency and Monetary use rank(method='first') to prevent bin edge errors on duplicate values
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=4, labels=[1, 2, 3, 4])

# Concatenate scores into a single segment string (e.g., '444' for top customers)
rfm['RFM_Segment'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

print("--- RFM Metrics Head ---")
print(rfm.head())

--- RFM Metrics Head ---
             Recency  Frequency  Monetary R_Score F_Score M_Score RFM_Segment
Customer ID                                                                  
12346.0          112         10    230.55       1       4       1         141
12349.0           35          2   1268.52       3       2       4         324
12355.0           32          1    488.21       3       1       2         312
12358.0           15          2   1697.93       4       2       4         424
12359.0           91          4   1522.23       2       4       4         244


In [3]:
# Define a function to categorize customers based on their RFM score
def segment_customer(score):
    if score == '444':
        return 'Champions'
    elif score.startswith('3') or score.startswith('4'):
        return 'Loyal Customers'
    elif score.startswith('1') and (score.endswith('3') or score.endswith('4')):
        return 'At Risk (High Value)'
    elif score.startswith('1') or score.startswith('2'):
        return 'Needs Attention / Lost'
    else:
        return 'Average Customers'

# Apply the function to create a new column
rfm['Customer_Profile'] = rfm['RFM_Segment'].apply(segment_customer)

print("--- RFM Segments Head ---")
print(rfm.head())

--- RFM Segments Head ---
             Recency  Frequency  Monetary R_Score F_Score M_Score RFM_Segment  \
Customer ID                                                                     
12346.0          112         10    230.55       1       4       1         141   
12349.0           35          2   1268.52       3       2       4         324   
12355.0           32          1    488.21       3       1       2         312   
12358.0           15          2   1697.93       4       2       4         424   
12359.0           91          4   1522.23       2       4       4         244   

                   Customer_Profile  
Customer ID                          
12346.0      Needs Attention / Lost  
12349.0             Loyal Customers  
12355.0             Loyal Customers  
12358.0             Loyal Customers  
12359.0      Needs Attention / Lost  


In [4]:
# Extract the month from the InvoiceDate
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')

# Find the first month each customer made a purchase (their Cohort)
df['CohortMonth'] = df.groupby('Customer ID')['InvoiceDate'].transform('min').dt.to_period('M')

# Calculate the difference in months between the invoice and the first purchase
df['CohortIndex'] = (df['InvoiceMonth'].dt.year - df['CohortMonth'].dt.year) * 12 + \
                    (df['InvoiceMonth'].dt.month - df['CohortMonth'].dt.month) + 1

# Count unique customers per cohort and month, then pivot the table
cohort_data = df.groupby(['CohortMonth', 'CohortIndex'])['Customer ID'].nunique().reset_index()
retention_matrix = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='Customer ID')

# Convert the raw customer counts into percentage retention rates
cohort_sizes = retention_matrix.iloc[:, 0]
retention_percentages = retention_matrix.divide(cohort_sizes, axis=0).round(3) * 100

print("--- Retention Percentages Head ---")
print(retention_percentages.head())

--- Retention Percentages Head ---
CohortIndex      1     2     3     4     5     6     7
CohortMonth                                           
2009-12      100.0  35.3  33.4  42.5  38.0  35.9  30.4
2010-01      100.0  20.6  31.1  30.5  26.4  23.5   NaN
2010-02      100.0  23.7  22.3  29.0  16.0   NaN   NaN
2010-03      100.0  19.0  23.0  18.7   NaN   NaN   NaN
2010-04      100.0  19.4  13.3   NaN   NaN   NaN   NaN


In [5]:
# Dataset volume audit

raw_rows = len(raw_df)

missing_customer_ids = raw_df["Customer ID"].isna().sum()

non_positive_quantity = (
    raw_df["Quantity"].isna() |
    (raw_df["Quantity"] <= 0)
).sum()

# Apply the same conditions as the SQL cleaning query
cleaned_mask = (
    raw_df["Customer ID"].notna() &
    (raw_df["Quantity"] > 0)
)

cleaned_rows = int(cleaned_mask.sum())

excluded_rows = raw_rows - cleaned_rows

retention_percentage = (
    cleaned_rows / raw_rows
) * 100

print("DATA QUALITY AUDIT")
print("-------------------")
print("Raw records:", raw_rows)
print("Missing Customer IDs:", missing_customer_ids)
print("Non-positive quantities:", non_positive_quantity)
print("Cleaned records:", cleaned_rows)
print("Excluded records:", excluded_rows)
print("Retention percentage:", round(retention_percentage, 2), "%")

DATA QUALITY AUDIT
-------------------
Raw records: 244618
Missing Customer IDs: 53165
Non-positive quantities: 6661
Cleaned records: 186406
Excluded records: 58212
Retention percentage: 76.2 %
